[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR-GITHUB-USERNAME/JAXCode/blob/master/solutions/07_lax_control_flow_solution.ipynb)

# 🟡 Solution: Newton's Method with lax.while_loop

*JAX Fundamentals · Medium*

Reference implementation. Try it yourself in `07_lax_control_flow.ipynb` first.

---
Implement $\sqrt{x}$ with **Newton's method**, using `jax.lax.while_loop` so it
runs inside `jit`.

$$g_{n+1} = \frac{1}{2}\left(g_n + \frac{x}{g_n}\right)$$

Iterate until $|g_{n+1}^2 - x| < \text{tol}$ or `max_iters` is reached.

### Rules
- Use `jax.lax.while_loop` — a Python `while` on a traced value raises
  `ConcretizationTypeError` under `jit`
- `x` is a **scalar**; the function must be `jit`-able and `vmap`-able
- Return `0.0` for `x == 0` (the update would divide by zero)
- Do not call `jnp.sqrt`, `x ** 0.5`, or `jnp.power`

### Signature
```python
def newton_sqrt(x, tol=1e-6, max_iters=50):
    ...
```

### Why it matters
`while_loop` is the escape hatch for **data-dependent** iteration counts — the
loop runs until a condition on traced values is met, which a Python loop cannot
express under tracing. The catch, and the thing interviewers probe: `while_loop`
is **not reverse-mode differentiable**, because the trip count is not known at
trace time. If you need gradients through an iterative solver, you either use
`lax.scan` with a fixed count, or `implicit differentiation` on the fixed point.

In [ ]:
# Install jax-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q jax-judge flax')
except ImportError:
    pass

In [ ]:
import jax
import jax.numpy as jnp

print("JAX", jax.__version__, "|", jax.devices())

In [ ]:
# ✅ REFERENCE SOLUTION

import jax
import jax.numpy as jnp


def newton_sqrt(x, tol=1e-6, max_iters=50):
    x = jnp.asarray(x, dtype=jnp.float32)

    def cond(carry):
        guess, i = carry
        return (jnp.abs(guess * guess - x) >= tol) & (i < max_iters)

    def body(carry):
        guess, i = carry
        return 0.5 * (guess + x / guess), i + 1

    # Start from x itself (any positive seed converges for x > 0).
    init = (jnp.maximum(x, 1.0), jnp.array(0))
    guess, _ = jax.lax.while_loop(cond, body, init)

    # x == 0 would divide by zero in the body, so special-case it.
    return jnp.where(x == 0, 0.0, guess)

In [ ]:
# 🔍 Verify
import jax
import jax.numpy as jnp

for v in [0.0, 1.0, 2.0, 16.0, 1e6]:
    print(f"newton_sqrt({v:>9}) = {newton_sqrt(v):.6f}   (jnp.sqrt = {jnp.sqrt(v):.6f})")

# Works under vmap, so you can do a whole array at once:
print(jax.vmap(newton_sqrt)(jnp.array([4.0, 9.0, 25.0])))

In [ ]:
# Run the judge against the reference solution
from jax_judge import check

check("lax_control_flow")